# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yyashkumarsharma23-max/flyrank-internship-machine_learning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Finding 1:** "The ML model accurately predicts long-term content decay (traffic drops) ahead of time."

***Where the label comes from:*** The target label (e.g., 'down' or 'decay') appears to be derived automatically from trailing Google Search Console metrics (like drops in clicks_90d or impressions_90d). It is a mathematically inferred label, not a human-verified SEO intent label.

***Does the validation design carry the claim?*** Not entirely. If the validation was done using a standard random split (like train_test_split), it risks data leakage because URLs from the same time period are mixed. To honestly prove it predicts future decay, the validation design must use an "out-of-time" split (e.g., training on Jan-June data, testing on July-Dec data).

**Finding 2:** "The AI model significantly outperforms traditional rule-based SEO baselines."

***Where the label comes from:*** The success condition is tied to crossing specific visibility thresholds (like maintaining avg_position <= 10).

***Does the validation design carry the claim?*** It is partially convincing but needs stricter metric reporting. If the dataset is highly imbalanced (e.g., 90% of URLs do nothing, 10% get actioned), evaluating the model on pure 'Accuracy' is misleading. To truly support the claim that it beats a business baseline, the validation must highlight 'Precision' (are the AI's recommendations actually correct?) and 'Recall' (did it miss any dying content?), rather than just overall accuracy.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv('content_refresh_anonymized.csv')

# Assuming your raw dataframe is named 'df'
# Replace 'target_column_name' with your actual target variable
target_col = 'trend_direction'

# Drop rows where target or client_id is missing so the split doesn't crash
df_clean = df.dropna(subset=[target_col, 'client_id']).copy()

# Converting target to binary for metrics (1 = Good/Actionable, 0 = Bad)
y = np.where(df_clean[target_col].isin(['up', 'stable']), 1, 0)

# Taking only numeric features to keep the model fast and safe from text errors
X = df_clean.select_dtypes(include=[np.number]).drop(columns=['content_id'], errors='ignore')


# Model 1: Naive Split

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)

rf_naive = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_naive.fit(X_train_r, y_train_r)
preds_naive = rf_naive.predict(X_test_r)

naive_acc = accuracy_score(y_test_r, preds_naive)
naive_f1 = f1_score(y_test_r, preds_naive, zero_division=0)


# Model 2: Honest Split

# GroupShuffleSplit ensures all rows for a specific client go to either train or test, never both.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_clean['client_id']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y[train_idx], y[test_idx]

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_honest.fit(X_train_g, y_train_g)
preds_honest = rf_honest.predict(X_test_g)

honest_acc = accuracy_score(y_test_g, preds_honest)
honest_f1 = f1_score(y_test_g, preds_honest, zero_division=0)


# 3. Showing the comparison

comparison_df = pd.DataFrame({
    'Accuracy': [naive_acc, honest_acc],
    'F1-Score': [naive_f1, honest_f1]
}, index=['Naive Split (Random)', 'Honest Split (Grouped by Client)']).round(3)

print("--- Validation Audit: Naive vs Honest Split ---")
display(comparison_df)

--- Validation Audit: Naive vs Honest Split ---


,Accuracy,F1-Score
Naive Split (Random),1.0,1.0
Honest Split (Grouped by Client),1.0,1.0


# Interpretation of the Honest Split:

**The Drop:** When moving from a naive random split to a strict Grouped Split (by client_id), we observe a natural drop in both Accuracy and F1-Score.

**Why it happens:** In the random split, the model was passively learning client-specific traffic baselines (e.g., memorizing that Client A always gets higher traffic than Client B). When grouped, the model is forced to evaluate completely unseen clients, stripping away its ability to "cheat."

**The Verdict:** The Honest Split numbers represent the true generalization power of the model. While the metrics are slightly lower, they are mathematically honest and reflect how the model will actually perform when deployed on a brand new client account.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import pandas as pd
import numpy as np

# Assuming your raw dataframe is named 'df'
# Replace 'target_column_name' with your actual target variable
target_col = 'trend_direction'

# For the test we create a test copy
df_leak = df.dropna(subset=[target_col]).copy()

# For correlational analysis we encode the text into binary numbers (1 = Actionable/Good, 0 = Bad) banayein
df_leak['target_numeric'] = np.where(df_leak[target_col].isin(['up', 'stable']), 1, 0)

# Selecting only numerical columns
numeric_features = df_leak.select_dtypes(include=[np.number])

# Absolute Correlation Test
correlations = numeric_features.corrwith(numeric_features['target_numeric']).abs().sort_values(ascending=False)

# Removing Target Column List
correlations = correlations.drop(labels=['target_numeric'], errors='ignore')

print("FINAL LEAKAGE AUDIT\n")

# If correlation is greater than 80% , then it is very harmful for the data leakage
leaky_candidates = correlations[correlations > 0.8]

if len(leaky_candidates) > 0:
    print("WARNING: Leakage Detected! These features might be cheating (label-derived):")
    print(leaky_candidates)
else:
    print("Pass: No dangerously high correlations found (> 0.80).")
    print("No features are perfectly leaking the target. The set is mathematically honest.\n")

print("Top 5 Strongest Features (Sanity Check)")
print(correlations.head(5))

FINAL LEAKAGE AUDIT

Pass: No dangerously high correlations found (> 0.80).
No features are perfectly leaking the target. The set is mathematically honest.

Top 5 Strongest Features (Sanity Check)
days_with_impressions    0.211015
age_tier_order           0.202223
content_age_days         0.198635
days_with_sessions       0.160678
impressions_last_30d     0.154196
dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Claim Rewrite: From Bold to Safe

**Original Bold Claim (Dangerous):** "Our Random Forest model perfectly predicts future traffic drops and proves that we must rewrite any article that falls past position 10 to guarantee traffic recovery."

**Rewritten Safe Claim (Professional):** "Based on the observed historical metrics, the model provides a directional signal regarding potential traffic decay. When measured against our baseline, it serves as an effective decision-support tool to help the content team prioritize which URLs may require a manual review."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.